In [1]:
from pathlib import Path
import json
import shutil
import numpy as np
import pandas as pd
import joblib
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score, average_precision_score
from IPython.display import display

SEEDS = [0, 1, 2]
N_TREES = 200
MAX_SAMPLES = 'auto'
INPUT_FILES = {'train': 'train_scaled.csv', 'val': 'val_scaled.csv', 'cal': 'cal_scaled.csv', 'dataset2': 'dataset2_eval.csv'}
search_dirs = [Path('/content/files'), Path('/content'), Path.cwd() / 'files', Path.cwd(), Path('data/processed'), Path('../../data/processed')]
DATA_DIR = next((p.resolve() for p in search_dirs if all((p / f).is_file() for f in INPUT_FILES.values())), None)
if DATA_DIR is None:
    raise FileNotFoundError('Upload train_scaled.csv, val_scaled.csv, cal_scaled.csv and dataset2_eval.csv into /content/ or /content/files/')
OUTPUT_DIR = Path('/content/iforest_ensemble_outputs') if Path('/content').is_dir() else Path('iforest_ensemble_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'models').mkdir(exist_ok=True)
(OUTPUT_DIR / 'scores').mkdir(exist_ok=True)
print('CSV directory:', DATA_DIR)
print('Output directory:', OUTPUT_DIR.resolve())

CSV directory: /content
Output directory: /content/iforest_ensemble_outputs


In [2]:
tables = {name: pd.read_csv(DATA_DIR / filename, parse_dates=['DATETIME']) for name, filename in INPUT_FILES.items()}
excluded = {'DATETIME', 'ATT_FLAG', 'label', 'attack_id'}
feature_names = [col for col in tables['train'].columns if col not in excluded]
assert len(feature_names) == 36 and sum(c.startswith('S_') for c in feature_names) == 8
assert [len(tables[x]) for x in INPUT_FILES] == [5256, 1728, 1729, 4177]
for split, df in tables.items():
    assert [col for col in df.columns if col not in excluded] == feature_names, f'{split}: mismatched features/order'
    assert pd.api.types.is_datetime64_any_dtype(df['DATETIME']) and df['DATETIME'].notna().all()
    assert df['DATETIME'].is_unique and df['DATETIME'].is_monotonic_increasing
    assert df['DATETIME'].diff().iloc[1:].eq(pd.Timedelta(hours=1)).all(), f'{split}: missing hourly rows'
    assert np.isfinite(df[feature_names].to_numpy(dtype=np.float64)).all(), f'{split}: invalid features'
    if split != 'dataset2':
        assert df['ATT_FLAG'].eq(0).all(), f'{split}: unexpectedly contains attacks'
for a, b in [('train', 'val'), ('val', 'cal')]:
    gap_hours = (tables[b]['DATETIME'].iloc[0] - tables[a]['DATETIME'].iloc[-1]) / pd.Timedelta(hours=1) - 1
    assert gap_hours >= 24, f'{a}/{b}: missing buffer gap'
assert {'label', 'attack_id'}.issubset(tables['dataset2'].columns)
assert tables['dataset2']['label'].isin([0, 1]).all()
assert int(tables['dataset2']['label'].sum()) == 492
assert (tables['dataset2']['label'].eq(1) == tables['dataset2']['attack_id'].ne(0)).all()
X = {split: df[feature_names].to_numpy(dtype=np.float64) for split, df in tables.items()}
labels = tables['dataset2']['label'].to_numpy(dtype=np.int64)
display(pd.DataFrame({'split': list(tables), 'rows': [len(tables[s]) for s in tables], 'features': len(feature_names)}))

,split,rows,features
0,train,5256,36
1,val,1728,36
2,cal,1729,36
3,dataset2,4177,36


In [3]:
score_tables = {split: pd.DataFrame({'DATETIME': df['DATETIME']}) for split, df in tables.items()}
metrics = []
for seed in SEEDS:
    model = IsolationForest(n_estimators=N_TREES, max_samples=MAX_SAMPLES, contamination='auto', random_state=seed, n_jobs=-1)
    model.fit(X['train'])
    model_path = OUTPUT_DIR / 'models' / f'iforest_seed{seed}.joblib'
    joblib.dump({'model': model, 'feature_names': feature_names, 'scoring': '-model.score_samples(X)', 'seed': seed}, model_path)
    for split in tables:
        score_tables[split][f'score_seed{seed}'] = -model.score_samples(X[split])
    dev_scores = score_tables['dataset2'][f'score_seed{seed}'].to_numpy()
    metrics.append({'seed': seed, 'roc_auc': roc_auc_score(labels, dev_scores), 'pr_auc_average_precision': average_precision_score(labels, dev_scores), 'n_estimators': N_TREES, 'train_rows': len(X['train'])})
    print(f'Seed {seed}: ROC-AUC {metrics[-1]["roc_auc"]:.4f} | PR-AUC {metrics[-1]["pr_auc_average_precision"]:.4f} | {model_path.name}')
for split, frame in score_tables.items():
    frame['score_mean'] = frame[[f'score_seed{s}' for s in SEEDS]].mean(axis=1)
    frame.to_csv(OUTPUT_DIR / 'scores' / f'iforest_scores_{split}.csv', index=False, date_format='%Y-%m-%d %H:%M:%S')
    assert len(frame) == len(tables[split]) and frame['DATETIME'].equals(tables[split]['DATETIME'])
metrics_df = pd.DataFrame(metrics)
mean_scores = score_tables['dataset2']['score_mean'].to_numpy()
metrics_df = pd.concat([metrics_df, pd.DataFrame([{'seed': 'mean_of_3', 'roc_auc': roc_auc_score(labels, mean_scores), 'pr_auc_average_precision': average_precision_score(labels, mean_scores), 'n_estimators': N_TREES, 'train_rows': len(X['train'])}])], ignore_index=True)
metrics_df.to_csv(OUTPUT_DIR / 'threshold_free_metrics.csv', index=False)
display(metrics_df)

Seed 0: ROC-AUC 0.6443 | PR-AUC 0.2119 | iforest_seed0.joblib
Seed 1: ROC-AUC 0.6455 | PR-AUC 0.2018 | iforest_seed1.joblib
Seed 2: ROC-AUC 0.6402 | PR-AUC 0.1967 | iforest_seed2.joblib


,seed,roc_auc,pr_auc_average_precision,n_estimators,train_rows
0,0,0.644295,0.211864,200,5256
1,1,0.645498,0.201772,200,5256
2,2,0.640225,0.196727,200,5256
3,mean_of_3,0.644459,0.203330,200,5256


In [4]:
manifest = {'model': 'sklearn.ensemble.IsolationForest', 'model_role': 'static unsupervised reference baseline', 'training_split': 'train_scaled.csv only', 'scoring': '-model.score_samples(X); larger means more anomalous', 'alarm_threshold': None, 'binary_predictions_exported': False, 'seeds': SEEDS, 'n_estimators': N_TREES, 'max_samples': MAX_SAMPLES, 'contamination': 'auto', 'feature_names_in_order': feature_names, 'score_columns': [f'score_seed{s}' for s in SEEDS] + ['score_mean'], 'score_files': {split: f'scores/iforest_scores_{split}.csv' for split in tables}, 'model_files': {str(s): f'models/iforest_seed{s}.joblib' for s in SEEDS}, 'dataset2_labels_used_for': 'threshold-independent evaluation only; not for fitting or selecting thresholds', 'dataset2_label_assumption': 'label=0 outside seven published attack windows, based on reconstructed evaluation schedule', 'ensemble_note': 'Align by DATETIME. Fit any ensemble score calibration or weights on an appropriate separate training/validation set; do not tune with Dataset 2 labels and then report Dataset 2 as independent test.'}
(OUTPUT_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2))
readme = """Isolation Forest — ensemble inputs

Run all cells with train_scaled.csv, val_scaled.csv, cal_scaled.csv, dataset2_eval.csv uploaded to Colab.

Model training: train_scaled.csv exclusively; three independently fitted 200-tree Isolation Forest models (random seeds 0, 1, 2).

Score definition: -IsolationForest.score_samples, higher indicates more anomalous; no alarm threshold or binary alarms are used.

Scores: scores/iforest_scores_train.csv, scores/iforest_scores_val.csv, scores/iforest_scores_cal.csv, scores/iforest_scores_dataset2.csv. Each file includes DATETIME, score_seed0, score_seed1, score_seed2, score_mean. Timestamps preserve source row order.

Models: models/iforest_seed0.joblib, iforest_seed1.joblib, iforest_seed2.joblib. Load with joblib.load(path)['model'] and pass features exactly in manifest.json order. Files require a trusted source and a compatible scikit-learn environment.

threshold_free_metrics.csv reports ROC-AUC and average precision on Dataset 2, using reconstructed labels solely for evaluation. No classification threshold, F1 or confusion matrix is reported here.

Ensemble: align by DATETIME; avoid training or tuning ensemble weights using Dataset 2 labels if Dataset 2 is also the evaluation set. Use val/cal inputs for score calibration without attacks; a separately labeled development set is needed for supervised ensemble weight fitting.
"""
(OUTPUT_DIR / 'README.txt').write_text(readme)
print('Saved:')
for path in sorted(OUTPUT_DIR.rglob('*')):
    if path.is_file():
        print(path.relative_to(OUTPUT_DIR), f'({path.stat().st_size:,} bytes)')

Saved:
README.txt (1,387 bytes)
manifest.json (1,822 bytes)
models/iforest_seed0.joblib (3,328,549 bytes)
models/iforest_seed1.joblib (3,262,725 bytes)
models/iforest_seed2.joblib (3,404,581 bytes)
scores/iforest_scores_cal.csv (167,171 bytes)
scores/iforest_scores_dataset2.csv (403,433 bytes)
scores/iforest_scores_train.csv (507,944 bytes)
scores/iforest_scores_val.csv (167,095 bytes)
threshold_free_metrics.csv (267 bytes)


In [5]:
archive = shutil.make_archive(str(OUTPUT_DIR), 'zip', root_dir=OUTPUT_DIR)
print('Download ZIP:', archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print('Not in Colab; ZIP saved to', archive)

Download ZIP: /content/iforest_ensemble_outputs.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>